In [15]:
import sys
from pathlib import Path
import geopandas as gpd
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import shutil

# Find the repository root.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, photo_paths
from src.prep import (
    download_photo,
    get_image_metadata,
    calculate_ground_extent,
    ground_to_mercator_extent,
    update_tracker,
)

cfg = load_config()

In [2]:
INDEX_PATH = cfg["paths"]["index"]
TRACKER_PATH = cfg["paths"]["tracker"]
LAYER = "photos"

print("Source index:", INDEX_PATH)
print("Batch tracker:", TRACKER_PATH)
print("County:", cfg["county"])
print("Years:", cfg["years"])
print("Scale:", cfg["scale"])

Source index: C:\Users\eu01251115\LocalProjects\batch_georef\data\main_index_blank.gpkg
Batch tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1957_tracker.gpkg
County: Carver
Years: [1957]
Scale: 20000


In [3]:
# Load the main index.
index_gdf = gpd.read_file(INDEX_PATH, layer=LAYER)

In [4]:
# examine all years in the county

county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

for year in sorted(county_gdf["year"].dropna().unique()):
    print(year)

1937
1938
1940
1947
1951
1953
1955
1956
1957
1960
1962
1963
1964
1967
1969
1971


### Manually determining ground scale:

- Visit the [MHAPO Air Photo Holdings page](https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery). 
- Select your chosen county to open a PDF with metadata. 
- Use this to determine the ground scale for your target year or years. Currently, this tool supports processing multiple years at the same time, but only if they share the same ground scale. 
- Set the scale for your currently batch in config.yml.
- Specify a batch name for the tracker table. The photos and the folders they are in will be named for the county and year. 

#### Once you have configured your year(s) and scale in config.yml, restart the kernel and start again from the beginning.


## Create a tracker table

In [5]:
# Select records for the configured county and years.
mask = (
    index_gdf["county"].str.casefold().eq(
        cfg["county"].casefold()
    )
    & index_gdf["year"].isin(cfg["years"])
)

In [6]:
batch_gdf = index_gdf.loc[mask].copy()

In [7]:
if batch_gdf.empty:
    raise ValueError("No photographs match this batch configuration.")

if (
    batch_gdf["cty_year_id"].isna().any()
    or not batch_gdf["cty_year_id"].is_unique
):
    raise ValueError("Batch contains missing or duplicate photo IDs.")

In [8]:
# check length and confirm CRS
print(f"Selected {len(batch_gdf)} photographs")
print("CRS:", batch_gdf.crs)

Selected 112 photographs
CRS: EPSG:3857


In [9]:
# Assign the configured scale to all selected photographs.
batch_gdf["scale"] = cfg["scale"]

In [10]:
# preview it
display(
    batch_gdf[
        ["cty_year_id", "county", "year", "scale", "url"]
    ].head()
)

,cty_year_id,county,year,scale,url
14201,carver_1957_wn-2t-43,Carver,1957,20000,https://geo.lib.umn.edu/hennepin_county/y1957/...
14202,carver_1957_wn-2t-44,Carver,1957,20000,https://geo.lib.umn.edu/hennepin_county/y1957/...
14203,carver_1957_wn-2t-41,Carver,1957,20000,https://geo.lib.umn.edu/hennepin_county/y1957/...
14204,carver_1957_wn-2t-2,Carver,1957,20000,https://geo.lib.umn.edu/hennepin_county/y1957/...
14205,carver_1957_wn-1t-174,Carver,1957,20000,https://geo.lib.umn.edu/hennepin_county/y1957/...


In [11]:
batch_gdf.columns

Index(['cty_year_id', 'status', 'county', 'year', 'photo_id', 'ground_width',
       'ground_height', 'wm_width', 'wm_height', 'scale', 'dpi_x', 'dpi_y',
       'width_px', 'height_px', 'worldfile_json', 'url', 'flight_code',
       'source_id', 'date_prepared', 'date_georef', 'gcp_count', 'confidence',
       'error', 'latitude', 'geometry'],
      dtype='str')

In [12]:
if TRACKER_PATH.exists():
    raise FileExistsError(
        f"Batch tracker already exists: {TRACKER_PATH}"
    )

TRACKER_PATH.parent.mkdir(parents=True, exist_ok=True)

batch_gdf.to_file(
    TRACKER_PATH,
    layer=LAYER,
    driver="GPKG",
)

print(f"Created batch tracker: {TRACKER_PATH}")

Created batch tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1957_tracker.gpkg


## Download JPEGS and calculate extents

In [ ]:
# Load the batch tracker for preparation
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

print(f"Batch: {cfg['batch_name']}")
print(f"Photographs: {len(tracker_gdf)}")

display(tracker_gdf.head())

Batch: carver_1957
Photographs: 112


,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1957_wn-2t-43,None,Carver,1957,wn-2t-43,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95944,None,None,None,None,None,44.820979,POINT (-10415339.431 5593382.186)
1,carver_1957_wn-2t-44,None,Carver,1957,wn-2t-44,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95945,None,None,None,None,None,44.821030,POINT (-10412903.983 5593390.189)
2,carver_1957_wn-2t-41,None,Carver,1957,wn-2t-41,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95980,None,None,None,None,None,44.848443,POINT (-10415506.855 5597693.411)
3,carver_1957_wn-2t-2,None,Carver,1957,wn-2t-2,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95981,None,None,None,None,None,44.877190,POINT (-10412481.414 5602208.241)
4,carver_1957_wn-1t-174,None,Carver,1957,wn-1t-174,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95993,None,None,None,None,None,44.905820,POINT (-10442744.397 5606706.938)


In [16]:
# Download original JPEGs.
download_results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Downloading photographs",
):
    photo_id = str(photo["cty_year_id"])
    jpeg_path = photo_paths(photo, cfg)["jpeg"]

    try:
        download_photo(photo["url"], jpeg_path)

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": None,
        })

    except Exception as exc:
        print(f"FAILED: {photo_id}: {exc}")

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": str(exc),
        })

download_df = pd.DataFrame(download_results)

print(f"Successful downloads: {download_df['error'].isna().sum()}")
print(f"Failed downloads: {download_df['error'].notna().sum()}")

display(download_df.head())

Successful downloads: 112
Failed downloads: 0


,cty_year_id,jpeg_path,error
0,carver_1957_wn-2t-43,C:\Users\eu01251115\LocalProjects\batch_georef...,None
1,carver_1957_wn-2t-44,C:\Users\eu01251115\LocalProjects\batch_georef...,None
2,carver_1957_wn-2t-41,C:\Users\eu01251115\LocalProjects\batch_georef...,None
3,carver_1957_wn-2t-2,C:\Users\eu01251115\LocalProjects\batch_georef...,None
4,carver_1957_wn-1t-174,C:\Users\eu01251115\LocalProjects\batch_georef...,None


In [17]:
# Collect preparation results for each photo.
results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Calculating extents",
):
    photo_id = photo["cty_year_id"]
    image_path = photo_paths(photo, cfg)["jpeg"]

    result = {
        "cty_year_id": photo_id,
        "width_px": None,
        "height_px": None,
        "dpi_x": None,
        "dpi_y": None,
        "ground_width": None,
        "ground_height": None,
        "wm_width": None,
        "wm_height": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # Read metadata from the existing JPEG.
        metadata = get_image_metadata(image_path)
        result.update(metadata)

        # Calculate ground dimensions in meters.
        result["ground_width"] = calculate_ground_extent(
            pixels=metadata["width_px"],
            dpi=metadata["dpi_x"],
            scale=photo["scale"],
        )

        result["ground_height"] = calculate_ground_extent(
            pixels=metadata["height_px"],
            dpi=metadata["dpi_y"],
            scale=photo["scale"],
        )

        # Convert dimensions to Web Mercator.
        result["wm_width"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude=photo["latitude"],
        )

        result["wm_height"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_height"],
            latitude=photo["latitude"],
        )

        # Mark preparation complete only after successful calculations.
        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date()

    except Exception as e:
        result["error"] = str(e)
        print(f"FAILED: {photo_id}: {e}")

    results.append(result)

# Convert results to a DataFrame for inspection.
results_df = pd.DataFrame(results)

print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Failed: {results_df['error'].notna().sum()}")

display(results_df.head())

Calculating extents: 100%|██████████| 112/112 [00:00<00:00, 2049.65it/s]

Successful: 112
Failed: 0


,cty_year_id,width_px,height_px,dpi_x,dpi_y,ground_width,ground_height,wm_width,wm_height,date_prepared,error
0,carver_1957_wn-2t-43,5433,5433,600,600,4599.940000,4599.940000,6485.066594,6485.066594,2026-11-12,None
1,carver_1957_wn-2t-44,5424,5424,600,600,4592.320000,4592.320000,6474.329527,6474.329527,2026-11-12,None
2,carver_1957_wn-2t-41,5436,5416,600,600,4602.480000,4585.546667,6491.740619,6467.856363,2026-11-12,None
3,carver_1957_wn-2t-2,5542,5496,600,600,4692.226667,4653.280000,6621.632747,6566.671522,2026-11-12,None
4,carver_1957_wn-1t-174,5390,5390,600,600,4563.533333,4563.533333,6443.228378,6443.228378,2026-11-12,None


In [18]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

Updated 112 records.


### Final load & preview

In [19]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1957_wn-2t-43,prepared,Carver,1957,wn-2t-43,4599.940000,4599.940000,6485.066594,6485.066594,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95944,2026-11-12,None,None,None,None,44.820979,POINT (-10415339.431 5593382.186)
1,carver_1957_wn-2t-44,prepared,Carver,1957,wn-2t-44,4592.320000,4592.320000,6474.329527,6474.329527,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95945,2026-11-12,None,None,None,None,44.821030,POINT (-10412903.983 5593390.189)
2,carver_1957_wn-2t-41,prepared,Carver,1957,wn-2t-41,4602.480000,4585.546667,6491.740619,6467.856363,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95980,2026-11-12,None,None,None,None,44.848443,POINT (-10415506.855 5597693.411)
3,carver_1957_wn-2t-2,prepared,Carver,1957,wn-2t-2,4692.226667,4653.280000,6621.632747,6566.671522,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95981,2026-11-12,None,None,None,None,44.877190,POINT (-10412481.414 5602208.241)
4,carver_1957_wn-1t-174,prepared,Carver,1957,wn-1t-174,4563.533333,4563.533333,6443.228378,6443.228378,20000,...,https://geo.lib.umn.edu/hennepin_county/y1957/...,Hennepin,95993,2026-11-12,None,None,None,None,44.905820,POINT (-10442744.397 5606706.938)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,carver_1957_wm-8t-28,prepared,Carver,1957,WM-8T-28,4657.513333,4635.500000,6572.782184,6541.716498,20000,...,https://geo.lib.umn.edu/Carver/y1957/WM-8T-28.jpg,Carver,100449,2026-11-12,None,None,None,None,44.878386,POINT (-10431147.8 5602396.127)
108,carver_1957_wm-8t-31,prepared,Carver,1957,WM-8T-31,4650.740000,4649.046667,6553.122926,6550.736936,20000,...,https://geo.lib.umn.edu/Carver/y1957/WM-8T-31.jpg,Carver,100450,2026-11-12,None,None,None,None,44.789629,POINT (-10449304.455 5588463.455)
109,carver_1957_wm-8t-33,prepared,Carver,1957,WM-8T-33,4650.740000,4635.500000,6553.152118,6531.678108,20000,...,https://geo.lib.umn.edu/Carver/y1957/WM-8T-33.jpg,Carver,100451,2026-11-12,None,None,None,None,44.789886,POINT (-10453874.12 5588503.785)
110,carver_1957_wm-8t-35,prepared,Carver,1957,WM-8T-35,4643.966667,4642.273333,6543.637263,6541.251251,20000,...,https://geo.lib.umn.edu/Carver/y1957/WM-8T-35.jpg,Carver,100452,2026-11-12,None,None,None,None,44.790143,POINT (-10458443.785 5588544.114)
